# Testing for Model Specifications: OLS and LM Tests

In [65]:
from libpysal import weights
import esda
import numpy as np 
import pandas as pd 
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns 
import contextily 
import spreg
import libpysal
from scipy import stats

In [3]:
# setting wd 
import os
os.chdir('/users/bkung/fooddesertproject')

In [45]:
# loading in data (beginning with Dallas)
dallas_data = gpd.read_file("modeling_data/dallas_modeling_final.gpkg")

In [53]:
# dropping NaNs 
variables_to_use = ['distance_to_nearest_uf', 
                 'distance_to_nearest_grocery',
                 'distance_to_nearest_fm',
                 'median_income',
                 'pct_no_vehicle',
                 'median_age',
                 'walking_ind',
                 'E_CHD'
                ]

dallas_data_clean = dallas_data.dropna(subset=variables_to_use).copy()

# logging income 
dallas_data_clean['log_income'] = np.log(dallas_data_clean['median_income'])

In [60]:
# generating spatial weights 
dallas_w = weights.distance.KNN.from_dataframe(dallas_data_clean, k=round(len(dallas_data_clean) ** (1/3)))
# row-standardization
dallas_w.transform = "R"

In [63]:
# Running OLS Model 
ind_variables = ['distance_to_nearest_uf', 
                 'distance_to_nearest_grocery',
                 'distance_to_nearest_fm',
                 'log_income',
                 'pct_no_vehicle',
                 'median_age',
                 'walking_ind'
                ]

dallas_ols = spreg.OLS(
    # Dependent variable
    dallas_data_clean[["E_CHD"]].values,
    # Independent variables
    dallas_data_clean[ind_variables].values,
    # Dependent variable name
    name_y="pct_chd",
    # Independent variable name
    name_x=ind_variables,
)
print(dallas_ols.summary)

# generating moran's residuals 
moran_residuals = spreg.MoranRes(dallas_ols, dallas_w, z=True)
print(f"Observed Moran's I:  {moran_residuals.I:.4f}")
print(f"Expected Moran's I:  {moran_residuals.eI:.4f}")
print(f"Standardized Z-score: {moran_residuals.zI:.4f}")
print(f"Analytical P-value:  {moran_residuals.p_norm:.4f}")

# generating LM tests 
lm_results = spreg.LMtests(dallas_ols, dallas_w)
print("LM Error p-value:", lm_results.lme[1])
print("LM Lag p-value:", lm_results.lml[1])

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :        None
Dependent Variable  :     pct_chd                Number of Observations:         659
Mean dependent var  :      5.8493                Number of Variables   :           8
S.D. dependent var  :      1.6959                Degrees of Freedom    :         651
R-squared           :      0.3564
Adjusted R-squared  :      0.3495
Sum squared residual:     1217.98                F-statistic           :     51.4949
Sigma-square        :       1.871                Prob(F-statistic)     :   2.294e-58
S.E. of regression  :       1.368                Log likelihood        :   -1137.468
Sigma-square ML     :       1.848                Akaike info criterion :    2290.936
S.E of regression ML:      1.3595                Schwarz criterion     :    2326.862

-----------------

In [67]:
variables_to_use = ['distance_to_nearest_uf', 
                 'distance_to_nearest_grocery',
                 'distance_to_nearest_fm',
                 'median_income',
                 'pct_no_vehicle',
                 'median_age',
                 'walking_ind',
                 'E_DIABETES'
                ]

dallas_data_clean = dallas_data.dropna(subset=variables_to_use).copy()
dallas_data_clean['log_income'] = np.log(dallas_data_clean['median_income'])

# generating spatial weights 
dallas_w = weights.distance.KNN.from_dataframe(dallas_data_clean, k=round(len(dallas_data_clean) ** (1/3)))
# row-standardization
dallas_w.transform = "R"

# Running OLS Model 
ind_variables = ['distance_to_nearest_uf', 
                 'distance_to_nearest_grocery',
                 'distance_to_nearest_fm',
                 'log_income',
                 'pct_no_vehicle',
                 'median_age',
                 'walking_ind'
                ]

dallas_ols = spreg.OLS(
    # Dependent variable
    dallas_data_clean[["E_DIABETES"]].values,
    # Independent variables
    dallas_data_clean[ind_variables].values,
    # Dependent variable name
    name_y="pct_diabetes",
    # Independent variable name
    name_x=ind_variables,
)
print(dallas_ols.summary)

# generating moran's residuals 
moran_residuals = spreg.MoranRes(dallas_ols, dallas_w, z=True)
print(f"Observed Moran's I:  {moran_residuals.I:.4f}")
print(f"Expected Moran's I:  {moran_residuals.eI:.4f}")
print(f"Standardized Z-score: {moran_residuals.zI:.4f}")
print(f"Analytical P-value:  {moran_residuals.p_norm:.4f}")

# generating LM tests 
lm_results = spreg.LMtests(dallas_ols, dallas_w)
print("LM Error p-value:", lm_results.lme[1])
print("LM Lag p-value:", lm_results.lml[1])

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ORDINARY LEAST SQUARES
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :        None
Dependent Variable  :pct_diabetes                Number of Observations:         659
Mean dependent var  :     12.9508                Number of Variables   :           8
S.D. dependent var  :      4.0703                Degrees of Freedom    :         651
R-squared           :      0.5045
Adjusted R-squared  :      0.4992
Sum squared residual:     5401.62                F-statistic           :     94.6886
Sigma-square        :       8.297                Prob(F-statistic)     :   5.797e-95
S.E. of regression  :       2.881                Log likelihood        :   -1628.259
Sigma-square ML     :       8.197                Akaike info criterion :    3272.519
S.E of regression ML:      2.8630                Schwarz criterion     :    3308.445

-----------------

It seems we should use SLX or Spatial Durbin.

In [69]:
# 3. Construct the spatially lagged independent variables (WX)
X_orig = dallas_data_clean[ind_variables].values
WX = libpysal.weights.lag_spatial(dallas_w, X_orig)

# Combine X and WX to form the structural matrix for the SDM/SLX models
X_sdm = np.hstack((X_orig, WX))
y = dallas_data_clean['E_CHD'].values

# ==========================================
# 4. Estimate the Unrestricted Model (SDM)
# ==========================================
# ML_Lag estimates: y = rho*W*y + X*B + W*X*G + e
sdm_model = spreg.ML_Lag(y, X_sdm, w=dallas_w)
loglik_sdm = sdm_model.logll

print(f"SDM Log-Likelihood: {loglik_sdm:.4f}")

# ==========================================
# 5. Estimate the Restricted Model (SLX)
# ==========================================
# To force ML_Lag to act as a pure SLX model, we bypass the MLE optimization
# loop and compute the log-likelihood manually with rho fixed strictly to 0.
n, k = X_sdm.shape

# Run an OLS regression on the SDM matrix to get the SLX coefficients
# Since rho=0, MLE simplifies down to OLS
X_with_intercept = np.hstack((np.ones((n, 1)), X_sdm))
beta_slx = np.linalg.lstsq(X_with_intercept, y, rcond=None)[0]

# Calculate residuals and residual sum of squares (RSS)
residuals_slx = y - np.dot(X_with_intercept, beta_slx)
rss_slx = np.sum(residuals_slx**2)
sigma2_slx = rss_slx / n

# Compute the exact log-likelihood for SLX under the ML framework (where rho=0)
# The Jacobian determinant ln|I - 0*W| = ln|I| = 0, so it drops out.
loglik_slx = - (n / 2) * np.log(2 * np.pi * sigma2_slx) - (rss_slx / (2 * sigma2_slx))

print(f"SLX Log-Likelihood: {loglik_slx:.4f}")

# ==========================================
# 6. Conduct the Likelihood Ratio (LR) Test
# ==========================================
# LR Statistic = 2 * (LogLik_Unrestricted - LogLik_Restricted)
lr_stat = 2 * (loglik_sdm - loglik_slx)

# Degrees of freedom = number of restrictions (we restricted exactly 1 parameter: rho = 0)
df_restriction = 1 

# Calculate the p-value from a Chi-Square distribution
p_value = 1 - stats.chi2.cdf(lr_stat, df_restriction)

print("\n--- Likelihood Ratio Test Results ---")
print(f"LR Statistic: {lr_stat:.4f}")
print(f"p-value:      {p_value:.5f}")

if p_value < 0.05:
    print("Conclusion: Reject H0. The spatial lag (rho) is significant. Use the Spatial Durbin Model (SDM).")
else:
    print("Conclusion: Fail to reject H0. The spatial lag (rho) is not significant. Safely use the SLX model.")


ML_Lag
SDM Log-Likelihood: -1007.5777
SLX Log-Likelihood: -1083.7252

--- Likelihood Ratio Test Results ---
LR Statistic: 152.2950
p-value:      0.00000
Conclusion: Reject H0. The spatial lag (rho) is significant. Use the Spatial Durbin Model (SDM).
